In [2]:
pip install numpy pandas matplotlib seaborn tqdm scipy statsmodels scikit-learn


   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.8 MB 9.0 MB/s eta 0:00:02
   --------- ------------------------------ 2.9/12.8 MB 7.5 MB/s eta 0:00:02
   ------------ --------------------------- 3.9/12.8 MB 6.6 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.8 MB 6.3 MB/s eta 0:00:02
   ---------------------- ----------------- 7.1/12.8 MB 6.8 MB/s eta 0:00:01
   ---------------------------- ----------- 9.2/12.8 MB 7.3 MB/s eta 0:00:01
   ---------------------------------- ----- 11.0/12.8 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.8 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 7.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.0 MB 9.9 MB/s eta 0:00:01
   ------------- -------------------------- 3.7/11.0 MB 9.6 MB/s eta 0:00:01
   -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# ==========================================================
# GSE33000 (VSCode/Windows)
#  - Series Matrix(발현) + SOFT(메타) 결합
#  - 라벨 자동탐지(AD/Control/HD) + NaN/∞ 클린업
#  - DEG(Welch t-test + FDR) + PCA/Volcano/Heatmap 저장
#  - 디버깅 파일: metadata_all.tsv / key_scores.tsv
# ==========================================================

import os, io, gzip, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
sns.set(rc={"figure.dpi":120})

# ===== 0) 경로 설정 (★본인 PC에 맞게 수정) =====
USER = "rhtmd"  # ← 사용자 폴더명
BASE = rf"C:\Users\{USER}\OneDrive\바탕 화면"  # ← OneDrive를 안 쓰면 일반 바탕화면 경로로 교체

MATRIX_PATH = rf"{BASE}\GSE33000_series_matrix.txt.gz"  # Series Matrix
SOFT_PATH   = rf"{BASE}\GSE33000_family.soft.gz"        # SOFT family
OUT_DIR     = rf"{BASE}\GSE33000_results"
os.makedirs(OUT_DIR, exist_ok=True)

def die(msg):
    print("[ERR]", msg); sys.exit(1)

# ===== 1) Series Matrix 읽기 =====
def load_series_matrix(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        die(f"Series Matrix 파일이 없습니다: {path}")
    with gzip.open(path, 'rt', errors='ignore', newline='') as f:
        lines = f.readlines()
    try:
        start = next(i for i,l in enumerate(lines) if l.startswith("!series_matrix_table_begin")) + 1
        end   = next(i for i,l in enumerate(lines) if l.startswith("!series_matrix_table_end"))
    except StopIteration:
        die("Series Matrix에서 table begin/end 구간을 찾지 못했습니다.")
    txt = "".join(lines[start:end])
    df = pd.read_csv(io.StringIO(txt), sep="\t", index_col=0)
    return df

# ===== 2) SOFT에서 메타데이터 추출 =====
def load_soft_metadata(path: str, sample_ids: list) -> pd.DataFrame:
    """
    SOFT family에서 각 GSM별 !Sample_* 값을 모두 모아 테이블화하고
    AD/Control/HD 패턴을 가장 잘 설명하는 열을 라벨로 선택.
    """
    if not os.path.exists(path):
        die(f"SOFT 파일이 없습니다: {path}")

    n = len(sample_ids)
    sample_set = set(sample_ids)
    meta = {}
    gsm = None

    # GSM별 !Sample_* 누적
    with gzip.open(path, 'rt', errors='ignore', newline='') as f:
        for line in f:
            if line.startswith("^SAMPLE"):
                gsm = line.strip().split(" = ", 1)[1]
                if gsm not in sample_set:
                    gsm = None
                    continue
                if gsm not in meta:
                    meta[gsm] = {}
            elif gsm and line.startswith("!Sample_"):
                parts = line.strip().split(" = ", 1)
                if len(parts) == 2:
                    k, v = parts[0], parts[1]
                    meta[gsm].setdefault(k, []).append(v)

    # GSM×key 테이블
    all_keys = sorted({k for d in meta.values() for k in d.keys()})
    tbl = pd.DataFrame(index=sample_ids, columns=all_keys, dtype=object)
    for gsm_id in sample_ids:
        row = meta.get(gsm_id, {})
        for k in all_keys:
            if k in row:
                tbl.loc[gsm_id, k] = " // ".join(row[k])

    # "key: value" 접두사 제거
    def strip_prefix(s):
        if not isinstance(s, str): return s
        parts = [p.strip() for p in s.split(" // ")]
        cleaned = [(p.split(":",1)[1].strip() if ":" in p else p) for p in parts]
        return " // ".join(cleaned)

    tbl_clean = tbl.applymap(strip_prefix)
    tbl_clean.to_csv(os.path.join(OUT_DIR, "metadata_all.tsv"), sep="\t")

    # 라벨 스코어링
    pat_AD  = re.compile(r"\b(alzheimer|ad)\b", flags=re.I)
    pat_CTL = re.compile(r"\b(control|non[-\s]?demented|nd)\b", flags=re.I)
    pat_HD  = re.compile(r"\b(huntington|hd)\b", flags=re.I)

    scores = {}
    for col in tbl_clean.columns:
        vals = tbl_clean[col].fillna("").astype(str).tolist()
        c_ad = sum(1 for v in vals if pat_AD.search(v))
        c_ct = sum(1 for v in vals if pat_CTL.search(v))
        c_hd = sum(1 for v in vals if pat_HD.search(v))
        scores[col] = (c_ad, c_ct, c_hd, c_ad + c_ct + c_hd)

    sc_df = pd.DataFrame.from_dict(
        {k: {"AD":a, "Control":c, "HD":h, "score":s} for k,(a,c,h,s) in scores.items()},
        orient="index"
    ).sort_values("score", ascending=False)
    sc_df.to_csv(os.path.join(OUT_DIR, "key_scores.tsv"), sep="\t")

    # 최적 열 선택 (우선순위 반영)
    priority = [
        "!Sample_characteristics_ch1", "!Sample_characteristics_ch2",
        "!Sample_characteristics_ch3", "!Sample_characteristics_ch1.1",
        "!Sample_source_name_ch1", "!Sample_title", "!Sample_description"
    ]
    cand = sc_df[sc_df["score"] > 0].index.tolist()
    if not cand:
        die("SOFT에서 AD/Control/HD 패턴을 가진 열을 찾지 못했습니다. metadata_all.tsv 확인 필요.")

    def col_rank(col):
        base = priority.index(col) if col in priority else 999
        return (base, -sc_df.loc[col, "score"])

    best_col = sorted(cand, key=col_rank)[0]
    print(f"[INFO] 라벨에 사용할 SOFT 키: {best_col} (score={int(sc_df.loc[best_col,'score'])})")

    labels_raw = tbl_clean[best_col].fillna("").astype(str)

    def norm_label(s):
        s_low = s.lower()
        if re.search(r"\b(alzheimer|ad)\b", s_low):   return "AD"
        if re.search(r"\b(huntington|hd)\b", s_low):  return "HD"
        if ("control" in s_low) or ("non-demented" in s_low) or re.search(r"\bnd\b", s_low):
            return "Control"
        return "Other"

    pheno = pd.DataFrame({"sample": sample_ids, "diagnosis_raw": labels_raw.values})
    pheno["group"] = pheno["diagnosis_raw"].map(norm_label)
    print("[INFO] 그룹 분포:\n", pheno["group"].value_counts())
    return pheno

# ===== 3) DEG + 그래프 (NaN/∞ 안전 클린업 포함) =====
def run_deg(expr_df: pd.DataFrame, pheno: pd.DataFrame, out_dir: str):
    use = pheno[pheno["group"].isin(["AD","Control"])]
    keep = [s for s in use["sample"].tolist() if s in expr_df.columns]
    if len(keep) < 10:
        die("AD/Control 샘플이 너무 적습니다. 라벨 매핑을 확인하세요.")

    # --- 서브매트릭스 & 숫자화 ---
    X = expr_df[keep].copy()
    X = X.apply(pd.to_numeric, errors="coerce")       # 문자열 → NaN
    X = X.replace([np.inf, -np.inf], np.nan)          # ±∞ → NaN
    X = X.dropna(axis=0, how="all")                   # 전부 NaN인 probe 제거
    X = X.dropna(axis=1, how="all")                   # 전부 NaN인 sample 제거(거의 없음)
    X = X.T.fillna(X.T.mean()).T                      # 행 평균 보간
    before = X.shape[0]
    X = X.dropna(axis=0, how="any")                   # 그래도 NaN 남은 행 제거
    removed = before - X.shape[0]
    if removed > 0:
        print(f"[INFO] 보간 후 제거된 probe 수: {removed}")

    labs = pheno.set_index("sample").loc[X.columns]["group"].values

    # --- PCA (QC) ---
    X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X.T)
    pca = PCA(n_components=2, random_state=42)
    PC = pca.fit_transform(X_std)
    pc_df = pd.DataFrame(PC, columns=["PC1","PC2"]); pc_df["group"] = labs
    plt.figure(figsize=(6,5))
    sns.scatterplot(data=pc_df, x="PC1", y="PC2", hue="group", s=25)
    plt.title(f"PCA (PC1 {pca.explained_variance_ratio_[0]*100:.1f}%, "
              f"PC2 {pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, "PCA.png")); plt.close()

    # --- Welch t-test + FDR ---
    mask_ad   = labs == "AD"
    mask_ctrl = labs == "Control"
    t_stat, pvals, logFC = [], [], []
    for probe in tqdm(X.index, desc="DE testing"):
        xa = X.loc[probe, mask_ad].values
        xc = X.loc[probe, mask_ctrl].values
        if np.isnan(xa).any() or np.isnan(xc).any():
            t_stat.append(np.nan); pvals.append(1.0); logFC.append(np.nan); continue
        t, p = ttest_ind(xa, xc, equal_var=False, nan_policy="omit")
        t_stat.append(t); pvals.append(p); logFC.append(np.nanmean(xa) - np.nanmean(xc))

    res = pd.DataFrame({"probe": X.index, "t": t_stat, "pval": pvals, "logFC": logFC}).set_index("probe")
    res["p_adj"] = multipletests(res["pval"].values, method="fdr_bh")[1]
    res.sort_values("p_adj", inplace=True)
    res.to_csv(os.path.join(out_dir, "DEG_AD_vs_Control.tsv"), sep="\t")
    print("[OK] DEG 저장:", os.path.join(out_dir, "DEG_AD_vs_Control.tsv"))

    # --- Volcano ---
    res["-log10(p_adj)"] = -np.log10(res["p_adj"] + 1e-300)
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=res, x="logFC", y="-log10(p_adj)", s=8, linewidth=0)
    sig = res["p_adj"] < 0.05
    plt.scatter(res.loc[sig,"logFC"], res.loc[sig,"-log10(p_adj)"], s=8)
    plt.axvline( 1.0, ls="--", c="gray"); plt.axvline(-1.0, ls="--", c="gray")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, "Volcano.png")); plt.close()

    # --- Heatmap (상위 50) ---
    topN = min(50, res.shape[0])
    top = res.head(topN).index
    mat = X.loc[top]
    mat_z = (mat - mat.mean(axis=1).values.reshape(-1,1)) / (mat.std(axis=1).values.reshape(-1,1) + 1e-9)
    sns.clustermap(mat_z, cmap="vlag", col_cluster=False, figsize=(8,10))
    plt.savefig(os.path.join(out_dir, "Heatmap.png")); plt.close()

# ===== 4) 메인 =====
def main():
    print("[INFO] 읽는 중:", MATRIX_PATH)
    expr_df = load_series_matrix(MATRIX_PATH)
    print("[INFO] Expression shape:", expr_df.shape)

    print("[INFO] SOFT에서 메타데이터 추출:", SOFT_PATH)
    pheno = load_soft_metadata(SOFT_PATH, expr_df.columns.tolist())

    run_deg(expr_df, pheno, OUT_DIR)
    print("[DONE] 결과 폴더:", OUT_DIR)

if __name__ == "__main__":
    main()


[INFO] 읽는 중: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_series_matrix.txt.gz
[INFO] Expression shape: (39280, 624)
[INFO] SOFT에서 메타데이터 추출: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_family.soft.gz


C:\Users\rhtmd\AppData\Local\Temp\ipykernel_27540\1816972076.py:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tbl_clean = tbl.applymap(strip_prefix)


[INFO] 라벨에 사용할 SOFT 키: !Sample_characteristics_ch2 (score=624)
[INFO] 그룹 분포:
 group
AD         310
Control    157
HD         157
Name: count, dtype: int64


DE testing: 100%|██████████| 39280/39280 [00:44<00:00, 888.82it/s]


[OK] DEG 저장: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\DEG_AD_vs_Control.tsv


c:\Users\rhtmd\AppData\Local\Programs\Python\Python313\Lib\site-packages\seaborn\matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)


[DONE] 결과 폴더: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results


In [13]:
pip install gseapy


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# ==========================================================
# GSE33000: probe-level DEG → gene-level 요약 + (가능 시) GSEA
#  - GPL4372 SOFT에서 플랫폼 테이블 스트리밍 파싱 (gz/텍스트 자동 판별)
#  - symbol 컬럼이 없으면 entrezgeneid로 폴백 (GSEA는 건너뜀)
#  - DEG TSV를 견고하게 로딩 (probe dtype 충돌/Unnamed:0 자동 처리)
#  - 병합 후 gene 대표 선택: (1) FDR 최소 → (2) |t| 최대
# ==========================================================

import os, io, re, gzip
import numpy as np
import pandas as pd

# ===== 경로(수정) =====
USER = "rhtmd"
BASE = rf"C:\Users\{USER}\OneDrive\바탕 화면"
OUT_DIR = rf"{BASE}\GSE33000_results"
GPL_PATH = rf"{BASE}\GPL4372_family.soft.gz"      # 확장자와 무관, gz 여부 자동 판별
DEG_PROBE_PATH = os.path.join(OUT_DIR, "DEG_AD_vs_Control.tsv")

os.makedirs(OUT_DIR, exist_ok=True)

# ---------- 공용 유틸 ----------
def is_gzip(path: str) -> bool:
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"

def open_text_maybe_gz(path: str):
    if is_gzip(path):
        return gzip.open(path, "rt", errors="ignore")
    return open(path, "rt", errors="ignore", encoding="utf-8", newline="")

# ---------- 1) GPL4372 플랫폼 테이블 로딩 (스트리밍) ----------
def load_gpl_table(gpl_soft_path: str) -> pd.DataFrame:
    if not os.path.exists(gpl_soft_path):
        raise FileNotFoundError(f"파일 없음: {gpl_soft_path}")
    buf = io.StringIO()
    inside = False
    with open_text_maybe_gz(gpl_soft_path) as f:
        for line in f:
            if line.startswith("!platform_table_begin"):
                inside = True
                continue
            if line.startswith("!platform_table_end"):
                break
            if inside:
                buf.write(line)
    content = buf.getvalue().strip()
    if not content:
        raise RuntimeError("GPL SOFT에서 platform_table 구간을 찾지 못했습니다.")
    df = pd.read_csv(io.StringIO(content), sep="\t", dtype=str)
    df.columns = [c.strip().lower() for c in df.columns]
    return df

gpl_df = load_gpl_table(GPL_PATH)

def pick_column(columns, patterns):
    for pat in patterns:
        hits = [c for c in columns if re.search(pat, c, re.I)]
        if hits:
            return hits[0]
    return None

# probe(ID) / symbol 탐지 (심볼 없으면 entrez로 폴백)
id_col  = pick_column(gpl_df.columns, [r"^(id|probe|reporter|spot_id|reporter_id)$", r"^id$"])
sym_col = pick_column(gpl_df.columns, [r"(gene.*symbol|symbol\b|genesymbol|gene_symbol)$", r"\bsymbol\b"])
id_type = "symbol" if sym_col else "entrez"
if not sym_col:
    sym_col = pick_column(gpl_df.columns, [r"^entrezgeneid$"])
    if not sym_col:
        raise RuntimeError(
            "심볼/entrez 컬럼 자동 탐지 실패.\n"
            f"columns={gpl_df.columns.tolist()}"
        )
if not id_col:
    raise RuntimeError(f"probe/ID 컬럼 자동 탐지 실패. columns={gpl_df.columns.tolist()}")

annot = gpl_df[[id_col, sym_col]].rename(columns={id_col:"probe", sym_col:"symbol"}).copy()

def clean_symbol(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip()
    if s in {"", "-", "NA", "N/A"}: return ""
    return re.split(r"[;,/|\s]+", s)[0].strip()

annot["symbol"] = annot["symbol"].map(clean_symbol)
annot = annot[(annot["symbol"]!="") & annot["probe"].notna()]
annot["probe"]  = annot["probe"].astype(str).str.strip().str.replace('"',"",regex=False)

# ---------- 2) DEG(probe-level) 견고 로더 ----------
def load_deg_probe_table(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"DEG 파일 없음: {path}")
    # 우선 전부 문자열로
    df = pd.read_csv(path, sep="\t", dtype=str)
    if "probe" not in df.columns:
        # 첫 컬럼이 인덱스 덤프인 경우
        first = df.columns[0]
        if first.startswith("Unnamed") or first.lower() in {"index","id"}:
            df = df.rename(columns={first:"probe"})
        else:
            # 정말 인덱스로 저장됐던 케이스
            df = pd.read_csv(path, sep="\t", index_col=0, dtype=str).reset_index()
            df = df.rename(columns={"index":"probe"})
    df["probe"] = df["probe"].astype(str).str.strip().str.replace('"',"",regex=False)
    # 통계 컬럼을 수치로 변환
    for c in ["t","pval","p_adj","logFC"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    missing = {"probe","t","p_adj"} - set(df.columns)
    if missing:
        raise RuntimeError(f"DEG 컬럼 누락: {missing} / columns={df.columns.tolist()}")
    return df

deg_probe = load_deg_probe_table(DEG_PROBE_PATH)

# ---------- 3) 병합 & gene 대표 선택 ----------
deg_merged = pd.merge(deg_probe, annot, on="probe", how="left").dropna(subset=["symbol"])
if deg_merged.empty:
    raise RuntimeError("어노테이션 병합 후 남은 행이 없습니다. GPL/DEG 파일을 확인하세요.")

deg_merged["rank_key"] = list(zip(deg_merged["p_adj"].values, -np.abs(deg_merged["t"].values)))
deg_gene = (deg_merged.sort_values("rank_key")
            .drop_duplicates(subset=["symbol"], keep="first")
            .drop(columns=["rank_key"])
            .set_index("symbol")
            .sort_values("p_adj"))

# 저장
if id_type == "symbol":
    gene_tsv = os.path.join(OUT_DIR, "DEG_AD_vs_Control_gene.tsv")
    deg_gene.to_csv(gene_tsv, sep="\t")
    print("[OK] gene-level DEG (symbol) →", gene_tsv)

    # ---------- 4) GSEA (심볼 기준) ----------
    rnk = os.path.join(OUT_DIR, "gsea_preranked.rnk")
    deg_gene[["t"]].rename(columns={"t":"score"}).to_csv(rnk, sep="\t", header=False)
    print("[OK] GSEA preranked 저장:", rnk)

    try:
        import gseapy as gp
        pr = gp.prerank(
            rnk=rnk,
            gene_sets="GO_Biological_Process_2021",
            min_size=15, max_size=500, permutation_num=100,
            outdir=OUT_DIR, seed=42, no_plot=False, format="png", verbose=True
        )
        print("[DONE] GSEA 완료 →", OUT_DIR)
    except ImportError:
        print("⚠ gseapy 미설치: `pip install gseapy` 후 재실행하세요.")

else:
    # 심볼이 없고 entrez로 대체된 경우
    gene_tsv = os.path.join(OUT_DIR, "DEG_AD_vs_Control_gene_by_entrez.tsv")
    rnk      = os.path.join(OUT_DIR, "gsea_preranked_entrez.rnk")
    deg_gene.to_csv(gene_tsv, sep="\t")
    deg_gene[["t"]].rename(columns={"t":"score"}).to_csv(rnk, sep="\t", header=False)
    print("[OK] gene-level DEG (entrez) →", gene_tsv)
    print("[OK] preranked (entrez) →", rnk)
    # 안내
    with open(os.path.join(OUT_DIR, "README_GSEA.txt"), "w", encoding="utf-8") as f:
        f.write(
            "이 플랫폼에는 gene symbol 컬럼이 없어 entrezgeneid를 사용했습니다.\n"
            "- 현재 rnk는 Entrez ID 기반입니다.\n"
            "- GSEA를 하려면 Entrez 기반 GMT를 쓰거나, Entrez→Symbol 매핑 후 symbol rnk를 생성하세요.\n"
        )
    print("⚠ 심볼 미존재로 GSEA는 건너뜀. README_GSEA.txt 참고.")


[OK] gene-level DEG (entrez) → C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\DEG_AD_vs_Control_gene_by_entrez.tsv
[OK] preranked (entrez) → C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\gsea_preranked_entrez.rnk
⚠ 심볼 미존재로 GSEA는 건너뜀. README_GSEA.txt 참고.


In [31]:
# ==========================================================
# Omics Feature Builder (GSE33000)
#  - Series Matrix(발현) + SOFT(라벨) + GPL4372(어노테이션)
#  - probe→gene collapse(심볼 없으면 entrez 폴백)
#  - Stratified train/val/test 분할(누수 방지)
#  - Train에서만 fit: z-score → 분산 상위 K 선택 → PCA
#  - 결과물 및 변환기(pkl/tsv/csv) 저장
# ==========================================================

import os, io, re, gzip, sys, json, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ====== 경로/하이퍼파라미터 (필요시 수정) ======
USER = "rhtmd"
BASE = rf"C:\Users\{USER}\OneDrive\바탕 화면"
MATRIX_PATH = rf"{BASE}\GSE33000_series_matrix.txt.gz"
SOFT_PATH   = rf"{BASE}\GSE33000_family.soft.gz"
GPL_PATH    = rf"{BASE}\GPL4372_family.soft.gz"
OUT_DIR     = rf"{BASE}\GSE33000_results\feature_build"
RANDOM_SEED = 42
TEST_SIZE   = 0.15
VAL_SIZE    = 0.15
TOPK_GENES  = 2000      # 분산 상위 K
N_PCA       = 128       # PCA 차원 수

os.makedirs(OUT_DIR, exist_ok=True)

# ====== 유틸 ======
def die(msg):
    print("[ERR]", msg); sys.exit(1)

def is_gzip(path: str) -> bool:
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"

def open_text_maybe_gz(path: str):
    if is_gzip(path):
        return gzip.open(path, "rt", errors="ignore")
    return open(path, "rt", errors="ignore", encoding="utf-8", newline="")

# ====== 1) Series Matrix 읽기 ======
def load_series_matrix(path: str) -> pd.DataFrame:
    if not os.path.exists(path): die(f"Series Matrix 없음: {path}")
    with gzip.open(path, 'rt', errors='ignore', newline='') as f:
        lines = f.readlines()
    try:
        s = next(i for i,l in enumerate(lines) if l.startswith("!series_matrix_table_begin")) + 1
        e = next(i for i,l in enumerate(lines) if l.startswith("!series_matrix_table_end"))
    except StopIteration:
        die("Series Matrix begin/end 구간을 찾지 못했습니다.")
    txt = "".join(lines[s:e])
    df = pd.read_csv(io.StringIO(txt), sep="\t", index_col=0)
    # 숫자화 + 결측 처리
    df = df.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=0, how="all")
    # 간단 보간 (probe별 평균으로)
    df = df.T.fillna(df.T.mean()).T
    df = df.dropna(axis=0, how="any")
    return df  # index: probe, columns: GSM IDs

# ====== 2) SOFT에서 라벨 추출 ======
def load_soft_labels(path: str, sample_ids: list) -> pd.DataFrame:
    if not os.path.exists(path): die(f"SOFT 없음: {path}")
    sample_set = set(sample_ids)
    meta = {}
    gsm = None
    with open_text_maybe_gz(path) as f:
        for line in f:
            if line.startswith("^SAMPLE"):
                gsm = line.strip().split(" = ", 1)[1]
                if gsm not in sample_set:
                    gsm = None
                    continue
                meta.setdefault(gsm, {})
            elif gsm and line.startswith("!Sample_"):
                parts = line.strip().split(" = ", 1)
                if len(parts)==2:
                    k,v = parts[0], parts[1]
                    meta[gsm].setdefault(k, []).append(v)
    # 테이블화
    all_keys = sorted({k for d in meta.values() for k in d.keys()})
    tbl = pd.DataFrame(index=sample_ids, columns=all_keys, dtype=object)
    for gsm_id in sample_ids:
        row = meta.get(gsm_id, {})
        for k in all_keys:
            if k in row:
                tbl.loc[gsm_id, k] = " // ".join(row[k])
    # 접두사 제거
    def strip_prefix(s):
        if not isinstance(s, str): return s
        parts = [p.strip() for p in s.split(" // ")]
        return " // ".join([(p.split(":",1)[1].strip() if ":" in p else p) for p in parts])
    tbl = tbl.applymap(strip_prefix)

    # 라벨 키 자동 선정
    pat_AD  = re.compile(r"\b(alzheimer|ad)\b", re.I)
    pat_CTL = re.compile(r"\b(control|non[-\s]?demented|nd)\b", re.I)
    pat_HD  = re.compile(r"\b(huntington|hd)\b", re.I)
    best_k, best_score = None, -1
    for k in tbl.columns:
        vals = tbl[k].fillna("").astype(str).tolist()
        c_ad = sum(1 for v in vals if pat_AD.search(v))
        c_ct = sum(1 for v in vals if pat_CTL.search(v))
        c_hd = sum(1 for v in vals if pat_HD.search(v))
        score = c_ad + c_ct + c_hd
        if score > best_score: best_k, best_score = k, score
    if best_score <= 0:
        die("SOFT에서 AD/Control/HD 패턴을 가진 열을 찾지 못했습니다.")

    raw = tbl[best_k].fillna("").astype(str)
    def norm(s):
        s = s.lower()
        if re.search(r"\b(alzheimer|ad)\b", s): return "AD"
        if re.search(r"\b(huntington|hd)\b", s): return "HD"
        if ("control" in s) or ("non-demented" in s) or re.search(r"\bnd\b", s): return "Control"
        return "Other"
    pheno = pd.DataFrame({"sample": sample_ids, "group": raw.map(norm)})
    return pheno

# ====== 3) GPL4372에서 probe→gene 매핑 ======
def load_gpl_table(path: str) -> pd.DataFrame:
    if not os.path.exists(path): die(f"GPL 없음: {path}")
    buf = io.StringIO(); inside=False
    with open_text_maybe_gz(path) as f:
        for line in f:
            if line.startswith("!platform_table_begin"): inside=True; continue
            if line.startswith("!platform_table_end"): break
            if inside: buf.write(line)
    content = buf.getvalue().strip()
    if not content: die("GPL platform_table 구간을 찾지 못했습니다.")
    df = pd.read_csv(io.StringIO(content), sep="\t", dtype=str)
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def build_probe_to_gene(gpl_df: pd.DataFrame):
    # symbol 우선, 없으면 entrez 폴백
    def pick(cols, pats):
        for pat in pats:
            hit = [c for c in cols if re.search(pat, c, re.I)]
            if hit: return hit[0]
        return None
    id_col  = pick(gpl_df.columns, [r"^(id|probe|reporter|spot_id|reporter_id)$", r"^id$"])
    sym_col = pick(gpl_df.columns, [r"(gene.*symbol|symbol\b|genesymbol|gene_symbol)$", r"\bsymbol\b"])
    id_type = "symbol" if sym_col else "entrez"
    if not sym_col:
        sym_col = pick(gpl_df.columns, [r"^entrezgeneid$"])
        if not sym_col:
            die(f"GPL에 symbol/entrez가 없습니다. columns={gpl_df.columns.tolist()}")
    if not id_col: die("GPL에 probe ID 컬럼을 찾지 못했습니다.")
    annot = gpl_df[[id_col, sym_col]].rename(columns={id_col:"probe", sym_col:"symbol"}).copy()

    def clean_symbol(s):
        if pd.isna(s): return ""
        s = str(s).strip()
        if s in {"", "-", "NA", "N/A"}: return ""
        return re.split(r"[;,/|\s]+", s)[0].strip()

    annot["symbol"] = annot["symbol"].map(clean_symbol)
    annot = annot[(annot["symbol"]!="") & annot["probe"].notna()]
    annot["probe"] = annot["probe"].astype(str).str.strip().str.replace('"',"",regex=False)
    return annot, id_type

# ====== 4) probe 표현행렬 → gene 표현행렬(collapse) ======
def collapse_probe_to_gene(expr_probe: pd.DataFrame, annot: pd.DataFrame, how: str="median") -> pd.DataFrame:
    # expr_probe: index=probe, columns=GSM
    expr = expr_probe.copy()
    expr.index = expr.index.astype(str).str.strip().str.replace('"',"",regex=False)
    expr = expr.loc[expr.index.intersection(annot["probe"])]
    # probe→gene 매핑
    m = annot.set_index("probe")["symbol"]
    genes = m.loc[expr.index]
    expr["__gene__"] = genes.values
    if how == "median":
        gene_expr = expr.groupby("__gene__").median(numeric_only=True)
    elif how == "mean":
        gene_expr = expr.groupby("__gene__").mean(numeric_only=True)
    else:
        # max variance probe 선택
        vari = expr.drop(columns="__gene__").var(axis=1)
        expr["__var__"] = vari
        expr = expr.sort_values(["__gene__","__var__"], ascending=[True, False])
        gene_expr = expr.drop_duplicates(subset="__gene__", keep="first").drop(columns=["__gene__","__var__"])
    gene_expr.index.name = "gene"
    return gene_expr

# ====== 5) 분할 → 표준화 → 분산 상위K → PCA ======
def split_stratified(samples, labels, test_size, val_size, seed=42):
    # 먼저 test 분할, 남은 것에서 val 분할
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx = np.arange(len(samples))
    tr_idx, te_idx = next(sss1.split(idx, labels))
    X_tr, y_tr = idx[tr_idx], labels[tr_idx]
    X_tmp, y_tmp = idx[te_idx], labels[te_idx]

    # val은 남은 집합에서
    if val_size > 0:
        val_rel = val_size / (1.0 - test_size)
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_rel, random_state=seed)
        tr2_idx, va_idx = next(sss2.split(X_tr, y_tr))
        tr_final = X_tr[tr2_idx]
        va_final = X_tr[va_idx]
    else:
        tr_final = X_tr
        va_final = np.array([], dtype=int)

    return tr_final, va_final, X_tmp

def build_features(gene_expr: pd.DataFrame, pheno: pd.DataFrame):
    # 샘플 정렬
    common = gene_expr.columns.intersection(pheno["sample"])
    gene_expr = gene_expr[common]
    ph = pheno.set_index("sample").loc[common]
    y = ph["group"].values
    samples = np.array(common)

    # AD/Control만
    keep_mask = np.isin(y, ["AD","Control"])
    samples = samples[keep_mask]; y = y[keep_mask]
    X = gene_expr[samples].T.values  # shape: [n_samples, n_genes]
    genes = gene_expr.index.to_numpy()

    # 분할
    tr_idx, va_idx, te_idx = split_stratified(samples, y, test_size=TEST_SIZE, val_size=VAL_SIZE, seed=RANDOM_SEED)

    def take(idx):
        return samples[idx], y[idx], X[idx, :]

    s_tr, y_tr, X_tr = take(tr_idx)
    s_va, y_va, X_va = take(va_idx) if len(va_idx)>0 else (np.array([]), np.array([]), np.empty((0, X.shape[1])))
    s_te, y_te, X_te = take(te_idx)

    # z-score (train에서 fit)
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_tr_z = scaler.fit_transform(X_tr)
    X_va_z = scaler.transform(X_va) if X_va.size else X_va
    X_te_z = scaler.transform(X_te)

    # 분산 상위 K (train에서 선택)
    var = X_tr_z.var(axis=0)
    idx_sorted = np.argsort(var)[::-1]
    k = min(TOPK_GENES, len(idx_sorted))
    top_idx = idx_sorted[:k]
    top_genes = genes[top_idx]

    X_tr_k = X_tr_z[:, top_idx]
    X_va_k = X_va_z[:, top_idx] if X_va.size else X_va_z
    X_te_k = X_te_z[:, top_idx]

    # PCA (train에서 fit)
    n_pca = min(N_PCA, X_tr_k.shape[1], X_tr_k.shape[0])  # 안전하게 축소
    pca = PCA(n_components=n_pca, random_state=RANDOM_SEED)
    Z_tr = pca.fit_transform(X_tr_k)
    Z_va = pca.transform(X_va_k) if X_va.size else X_va_k
    Z_te = pca.transform(X_te_k)

    # 저장물 구성
    artifacts = {
        "genes_selected": top_genes.tolist(),
        "scaler": scaler,
        "pca": pca,
        "label_map": {"AD":1, "Control":0},
        "splits": {
            "train_ids": s_tr.tolist(),
            "val_ids": s_va.tolist(),
            "test_ids": s_te.tolist()
        },
        "config": {
            "TOPK_GENES": TOPK_GENES,
            "N_PCA": int(n_pca),
            "TEST_SIZE": TEST_SIZE,
            "VAL_SIZE": VAL_SIZE,
            "RANDOM_SEED": RANDOM_SEED
        }
    }
    # 표로 저장
    def to_df(Z, ids, labs):
        df = pd.DataFrame(Z, index=ids, columns=[f"PC{i+1}" for i in range(Z.shape[1])])
        df.insert(0, "label", labs)
        return df

    df_tr = to_df(Z_tr, s_tr, y_tr)
    df_va = to_df(Z_va, s_va, y_va) if Z_va.size else pd.DataFrame()
    df_te = to_df(Z_te, s_te, y_te)

    return df_tr, df_va, df_te, artifacts

# ====== 메인 ======
def main():
    print("[INFO] Load Series Matrix...")
    expr_probe = load_series_matrix(MATRIX_PATH)

    print("[INFO] Parse labels from SOFT...")
    pheno = load_soft_labels(SOFT_PATH, expr_probe.columns.tolist())

    print("[INFO] Load GPL and build probe→gene mapping...")
    gpl_df = load_gpl_table(GPL_PATH)
    annot, id_type = build_probe_to_gene(gpl_df)
    if id_type == "entrez":
        print("⚠ 심볼 컬럼 없음 → entrez ID로 collapse 진행")

    print("[INFO] Collapse probe→gene (median)...")
    gene_expr = collapse_probe_to_gene(expr_probe, annot, how="median")
    gene_expr.to_csv(os.path.join(OUT_DIR, "gene_expression_matrix.tsv"), sep="\t")

    print("[INFO] Build features with leakage-safe pipeline...")
    df_tr, df_va, df_te, arts = build_features(gene_expr, pheno)

    # 저장
    df_tr.to_csv(os.path.join(OUT_DIR, "features_train.csv"), index=True)
    if not df_va.empty:
        df_va.to_csv(os.path.join(OUT_DIR, "features_val.csv"), index=True)
    df_te.to_csv(os.path.join(OUT_DIR, "features_test.csv"), index=True)

    # 변환기/선택 유전자 저장
    with open(os.path.join(OUT_DIR, "artifacts.pkl"), "wb") as f:
        pickle.dump(arts, f)
    with open(os.path.join(OUT_DIR, "artifacts.json"), "w", encoding="utf-8") as f:
        json.dump(
            {k:(v if k!="scaler" and k!="pca" else f"<<saved in artifacts.pkl>>")
             for k,v in arts.items()},
            f, ensure_ascii=False, indent=2
        )

    print("[DONE] Saved to:", OUT_DIR)
    print(" - features_train.csv / features_val.csv / features_test.csv")
    print(" - gene_expression_matrix.tsv")
    print(" - artifacts.pkl (scaler, pca, selected genes, splits)")
    print(" - artifacts.json (요약)")

if __name__ == "__main__":
    main()


[INFO] Load Series Matrix...
[INFO] Parse labels from SOFT...


C:\Users\rhtmd\AppData\Local\Temp\ipykernel_27540\3237544005.py:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tbl = tbl.applymap(strip_prefix)


[INFO] Load GPL and build probe→gene mapping...
⚠ 심볼 컬럼 없음 → entrez ID로 collapse 진행
[INFO] Collapse probe→gene (median)...
[INFO] Build features with leakage-safe pipeline...
[DONE] Saved to: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\feature_build
 - features_train.csv / features_val.csv / features_test.csv
 - gene_expression_matrix.tsv
 - artifacts.pkl (scaler, pca, selected genes, splits)
 - artifacts.json (요약)


In [32]:
import os, pandas as pd, numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score

# === 경로 수정 ===
USER = "rhtmd"
BASE = rf"C:\Users\{USER}\OneDrive\바탕 화면\GSE33000_results\feature_build"

tr_path = os.path.join(BASE, "features_train.csv")
va_path = os.path.join(BASE, "features_val.csv")
te_path = os.path.join(BASE, "features_test.csv")

def load_split(path):
    df = pd.read_csv(path, index_col=0)
    y  = (df["label"].astype(str).str.upper() == "AD").astype(int).values
    X  = df.drop(columns=["label"]).values
    ids= df.index.astype(str).tolist()
    return X, y, ids

Xtr, ytr, idtr = load_split(tr_path)
Xva, yva, idva = load_split(va_path) if os.path.exists(va_path) else (np.empty((0, Xtr.shape[1])), np.array([]), [])
Xte, yte, idte = load_split(te_path)

# --- 베이스라인 MLP ---
clf = MLPClassifier(hidden_layer_sizes=(256,128), activation="relu",
                    batch_size=32, learning_rate_init=1e-3,
                    max_iter=200, random_state=42, verbose=False)

clf.fit(Xtr, ytr)
p_tr = clf.predict_proba(Xtr)[:,1]
p_va = clf.predict_proba(Xva)[:,1] if Xva.size else np.array([])
p_te = clf.predict_proba(Xte)[:,1]

# --- 검증셋으로 플랫(시그모이드) 보정 ---
if Xva.size:
    cal = CalibratedClassifierCV(clf, method="sigmoid", cv="prefit")
    cal.fit(Xva, yva)
    p_tr = cal.predict_proba(Xtr)[:,1]
    p_va = cal.predict_proba(Xva)[:,1]
    p_te = cal.predict_proba(Xte)[:,1]
else:
    cal = None

# --- 지표 출력 ---
def prn(tag, y, p):
    if p.size==0: return
    auc = roc_auc_score(y, p)
    ap  = average_precision_score(y, p)
    print(f"{tag:>6} | AUC={auc:.4f}  AP={ap:.4f}")

prn("TRAIN", ytr, p_tr)
prn("VALID", yva, p_va)
prn("TEST ", yte, p_te)

# --- 확률 CSV 저장 (앙상블 입력용) ---
out_dir = os.path.join(BASE, "preds_omics"); os.makedirs(out_dir, exist_ok=True)
pd.DataFrame({"id":idtr, "label":ytr, "prob_omics":p_tr}).to_csv(os.path.join(out_dir,"train_probs.csv"), index=False)
if Xva.size:
    pd.DataFrame({"id":idva, "label":yva, "prob_omics":p_va}).to_csv(os.path.join(out_dir,"val_probs.csv"), index=False)
pd.DataFrame({"id":idte, "label":yte, "prob_omics":p_te}).to_csv(os.path.join(out_dir,"test_probs.csv"), index=False)

print("[OK] Saved omics probabilities to:", out_dir)


 TRAIN | AUC=1.0000  AP=1.0000
 VALID | AUC=0.9801  AP=0.9902
 TEST  | AUC=0.9787  AP=0.9884
[OK] Saved omics probabilities to: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\feature_build\preds_omics


c:\Users\rhtmd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [35]:
pip install torch scikit-learn pandas numpy


   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 1.6/241.3 MB 8.1 MB/s eta 0:00:30
    --------------------------------------- 3.7/241.3 MB 8.9 MB/s eta 0:00:27
    --------------------------------------- 5.2/241.3 MB 8.8 MB/s eta 0:00:27
   - -------------------------------------- 7.1/241.3 MB 8.6 MB/s eta 0:00:28
   - -------------------------------------- 8.9/241.3 MB 8.8 MB/s eta 0:00:27
   - -------------------------------------- 10.7/241.3 MB 8.9 MB/s eta 0:00:27
   -- ------------------------------------- 12.6/241.3 MB 8.9 MB/s eta 0:00:26
   -- ------------------------------------- 14.7/241.3 MB 9.0 MB/s eta 0:00:26
   -- ------------------------------------- 16.0/241.3 MB 8.7 MB/s eta 0:00:26
   -- ------------------------------------- 18.1/241.3 MB 8.8 MB/s eta 0:00:26
   --- ------------------------------------ 19.9/241.3 MB 8.9 MB/s eta 0:00:25
   --- ------------------------------------ 21.8/241.3 MB 8.9 MB/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
# ==========================================================
# Omics MLP: 학습/평가/온도보정/저장 (PyTorch)
# - 입력: features_{train,val,test}.csv  (label + PC1..PCn)
# - 출력:
#   * models_mlp/omics_mlp.pt           (가중치)
#   * models_mlp/temperature.json       (온도보정 T)
#   * preds_omics/{train,val,test}_probs.csv  (id,label,prob_omics)
#   * metrics_mlp.json                  (AUC/PR-AUC/F1/NLL)
# ==========================================================
import os, json, random, argparse
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, log_loss
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ---------- 설정 ----------
USER = "rhtmd"
BASE = rf"C:\Users\{USER}\OneDrive\바탕 화면\GSE33000_results\feature_build"
OUT_DIR = os.path.join(BASE, "models_mlp")
PRED_DIR = os.path.join(BASE, "preds_omics")  # 앙상블에서 재사용

CFG = dict(
    seed=42,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=300,
    patience=20,          # Early stopping on val AUC
    dropout=0.4,
    hidden=[128,128,64],  # MLP 구조
    pos_weight=None,      # 불균형 심할 때 예: 1.5
    device="cuda" if torch.cuda.is_available() else "cpu",
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

# ---------- 유틸 ----------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def load_split(path):
    df = pd.read_csv(path, index_col=0)
    ids = df.index.astype(str).tolist()
    y   = (df["label"].astype(str).str.upper()=="AD").astype(int).values
    X   = df.drop(columns=["label"]).values.astype(np.float32)
    return ids, X, y

def metrics_bin(y_true, p_prob):
    y_pred = (p_prob >= 0.5).astype(int)
    out = {
        "AUC": float(roc_auc_score(y_true, p_prob)),
        "AP":  float(average_precision_score(y_true, p_prob)),
        "F1":  float(f1_score(y_true, y_pred)),
        "NLL": float(log_loss(y_true, np.clip(p_prob,1e-6,1-1e-6)))
    }
    return out

class TabDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y.astype(np.float32))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, dropout=0.4):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, 1)]  # logits
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)  # logits

@torch.no_grad()
def predict_probs(model, loader, device):
    model.eval(); probs=[]; labels=[]
    for xb, yb in loader:
        xb = xb.to(device)
        logit = model(xb)
        prob  = torch.sigmoid(logit).cpu().numpy()
        probs.append(prob); labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logit = model(xb)
        loss = criterion(logit, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)

# Temperature scaling (val NLL 최소화)
class _Temp(nn.Module):
    def __init__(self, T=1.0):
        super().__init__()
        self.T = nn.Parameter(torch.tensor(float(T)))

def fit_temperature(model, loader_val, device, max_iter=200, lr=0.01):
    model.eval()
    temp = _Temp(T=1.0).to(device)
    opt  = optim.LBFGS(temp.parameters(), lr=lr, max_iter=max_iter, line_search_fn="strong_wolfe")

    bce = nn.BCEWithLogitsLoss(reduction="mean")
    # 고정된 로짓/레이블 캐시
    logits_list, labels_list = [], []
    with torch.no_grad():
        for xb, yb in loader_val:
            xb, yb = xb.to(device), yb.to(device)
            logits_list.append(model(xb))
            labels_list.append(yb)
    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list)

    def _closure():
        opt.zero_grad()
        loss = bce(logits / temp.T.clamp_min(1e-3), labels)
        loss.backward()
        return loss

    opt.step(_closure)
    T_star = float(temp.T.data.detach().cpu().item())
    return max(T_star, 1e-3)  # 안전 클램프

def apply_temperature(logits, T):
    return logits / max(T, 1e-3)

# ---------- 메인 ----------
def main():
    set_seed(CFG["seed"])

    tr_path = os.path.join(BASE, "features_train.csv")
    va_path = os.path.join(BASE, "features_val.csv")
    te_path = os.path.join(BASE, "features_test.csv")

    ids_tr, Xtr, ytr = load_split(tr_path)
    has_val = os.path.exists(va_path)
    if has_val:
        ids_va, Xva, yva = load_split(va_path)
    else:
        ids_va, Xva, yva = [], np.empty((0, Xtr.shape[1]), np.float32), np.array([], int)
    ids_te, Xte, yte = load_split(te_path)

    in_dim = Xtr.shape[1]
    model  = MLP(in_dim, CFG["hidden"], CFG["dropout"]).to(CFG["device"])

    # 손실: BCEWithLogits (pos_weight 선택적)
    if CFG["pos_weight"] is not None:
        pos_w = torch.tensor([float(CFG["pos_weight"])], device=CFG["device"])
    else:
        # 자동 pos_weight (선택): class imbalance 완화
        pos_ratio = (ytr==1).mean()
        pos_w = torch.tensor([max(1e-6, (1-pos_ratio)/max(pos_ratio,1e-6))], device=CFG["device"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    optimizer = optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

    dl_tr = DataLoader(TabDS(Xtr, ytr), batch_size=CFG["batch_size"], shuffle=True, drop_last=False)
    dl_va = DataLoader(TabDS(Xva, yva), batch_size=CFG["batch_size"], shuffle=False, drop_last=False) if has_val else None
    dl_te = DataLoader(TabDS(Xte, yte), batch_size=CFG["batch_size"], shuffle=False, drop_last=False)

    # ---- 학습 루프 (Early stopping: Val AUC) ----
    best_auc, best_epoch, best_state = -1.0, -1, None
    wait = 0
    for epoch in range(1, CFG["max_epochs"]+1):
        tr_loss = train_one_epoch(model, dl_tr, criterion, optimizer, CFG["device"])

        # 평가
        with torch.no_grad():
            # train
            p_tr, _ = predict_probs(model, dl_tr, CFG["device"])
            m_tr = metrics_bin(ytr, p_tr)
            # val
            if has_val:
                p_va, _ = predict_probs(model, dl_va, CFG["device"])
                m_va = metrics_bin(yva, p_va)
                monitor = m_va["AUC"]
            else:
                # val 없으면 train AUC로 모니터(비권장)
                monitor = m_tr["AUC"]; m_va = {"AUC":np.nan,"AP":np.nan,"F1":np.nan,"NLL":np.nan}

        if monitor > best_auc:
            best_auc, best_epoch, best_state = monitor, epoch, {k:v.cpu().clone() for k,v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"[E{epoch:03d}] loss={tr_loss:.4f} | AUC(tr)={m_tr['AUC']:.4f} AUC(va)={m_va['AUC']:.4f} | best={best_auc:.4f}@{best_epoch}")

        if has_val and wait >= CFG["patience"]:
            print(f"[EarlyStop] epoch={epoch}, best_auc={best_auc:.4f} @ {best_epoch}")
            break

    # best 모델 로드
    if best_state is not None:
        model.load_state_dict({k:v.to(CFG["device"]) for k,v in best_state.items()})

    # ---- 온도보정 (val 로 NLL 최소화) ----
    if has_val and len(yva) > 0:
        T_star = fit_temperature(model, dl_va, CFG["device"])
    else:
        T_star = 1.0
    with open(os.path.join(OUT_DIR, "temperature.json"), "w") as f:
        json.dump({"T": T_star}, f)

    # ---- 최종 평가 및 확률 저장 ----
    def eval_and_save(split, ids, X, y):
        dl = DataLoader(TabDS(X, y), batch_size=CFG["batch_size"], shuffle=False)
        model.eval()
        logits, labels = [], []
        with torch.no_grad():
            for xb, yb in dl:
                xb = xb.to(CFG["device"])
                lg = model(xb)
                lg = apply_temperature(lg, T_star)  # 온도 적용
                logits.append(lg.cpu().numpy())
                labels.append(yb.numpy())
        logits = np.concatenate(logits); labels = np.concatenate(labels)
        probs = 1.0 / (1.0 + np.exp(-logits))
        # 지표
        m = metrics_bin(labels, probs)
        # 저장
        out_path = os.path.join(PRED_DIR, f"{split}_probs.csv")
        pd.DataFrame({"id": ids, "label": labels.astype(int), "prob_omics": probs}).to_csv(out_path, index=False)
        return m, out_path

    mt, pt = eval_and_save("train", ids_tr, Xtr, ytr)
    mv, pv = (eval_and_save("val", ids_va, Xva, yva) if has_val and len(yva)>0 else ({"AUC":None,"AP":None,"F1":None,"NLL":None}, None))
    mte, pte = eval_and_save("test", ids_te, Xte, yte)

    # ---- 모델 저장 ----
    torch.save({
        "state_dict": model.state_dict(),
        "config": CFG,
        "in_dim": in_dim,
        "hidden": CFG["hidden"],
        "dropout": CFG["dropout"],
        "temperature": T_star
    }, os.path.join(OUT_DIR, "omics_mlp.pt"))

    # ---- 메트릭 요약 저장 ----
    metrics_all = {"train": mt, "val": mv[0] if isinstance(mv, tuple) else mv, "test": mte}
    with open(os.path.join(OUT_DIR, "metrics_mlp.json"), "w") as f:
        json.dump(metrics_all, f, indent=2)
    print("\n[RESULT] metrics:", json.dumps(metrics_all, indent=2))
    print("[OK] Saved model to:", os.path.join(OUT_DIR, "omics_mlp.pt"))
    print("[OK] Saved probs to:", PRED_DIR)

if __name__ == "__main__":
    main()


[E001] loss=0.4290 | AUC(tr)=0.5495 AUC(va)=0.8958 | best=0.8958@1
[E010] loss=0.1525 | AUC(tr)=0.4601 AUC(va)=0.9710 | best=0.9710@10
[E020] loss=0.0577 | AUC(tr)=0.5093 AUC(va)=0.9819 | best=0.9819@20
[E030] loss=0.0420 | AUC(tr)=0.5208 AUC(va)=0.9737 | best=0.9819@20
[E040] loss=0.0181 | AUC(tr)=0.5241 AUC(va)=0.9755 | best=0.9819@20
[EarlyStop] epoch=40, best_auc=0.9819 @ 20

[RESULT] metrics: {
  "train": {
    "AUC": 0.9999577220648544,
    "AP": 0.999978861032427,
    "F1": 0.9953703703703703,
    "NLL": 0.038292380229715726
  },
  "val": {
    "AUC": 0.9818840579710145,
    "AP": 0.9903724219401324,
    "F1": 0.9213483146067416,
    "NLL": 0.20039751371689526
  },
  "test": {
    "AUC": 0.9778368794326241,
    "AP": 0.9871522988408284,
    "F1": 0.9574468085106383,
    "NLL": 0.1629882689847612
  }
}
[OK] Saved model to: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\feature_build\models_mlp\omics_mlp.pt
[OK] Saved probs to: C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\featur

In [1]:
# ==========================================================
# AD vs Control 평가표 생성 (테스트셋)
#  - 입력: feature_build/preds_omics/{val,test}_probs.csv
#          (컬럼: id, label(0/1), prob_omics)
#  - 절차:
#     1) 검증셋으로 최적 임계값(threshold) 추정
#        - Youden J (sens+spec-1) 기본
#        - (참고) F1 최적 임계값도 계산
#     2) 테스트셋에 임계값 적용 → 혼동행렬/지표 표 생성
#  - 출력:
#     - evaluation/thresholds.txt
#     - evaluation/confusion_matrix_test.csv   (2x2 표)
#     - evaluation/metrics_test.csv            (요약 지표 표)
#     - evaluation/per_sample_test.csv         (샘플별 예측 표)
# ==========================================================

import os, numpy as np, pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss,
    confusion_matrix, precision_recall_fscore_support, matthews_corrcoef, accuracy_score
)

# === 경로 수정 ===
USER = "rhtmd"
BASE = rf"C:\Users\{USER}\OneDrive\바탕 화면\GSE33000_results\feature_build"
PRED_DIR = os.path.join(BASE, "preds_omics")   # 기본: 오믹스 확률
OUT_DIR  = os.path.join(BASE, "evaluation")
os.makedirs(OUT_DIR, exist_ok=True)

VAL_CSV  = os.path.join(PRED_DIR, "val_probs.csv")
TEST_CSV = os.path.join(PRED_DIR, "test_probs.csv")

# === 유틸 ===
def load_probs(path, prob_col="prob_omics"):
    df = pd.read_csv(path)
    if not {"id","label",prob_col}.issubset(df.columns):
        raise RuntimeError(f"필수 컬럼 누락: {path} → {df.columns.tolist()}")
    y = df["label"].astype(int).values  # 1=AD, 0=Control
    p = df[prob_col].astype(float).values
    ids = df["id"].astype(str).values
    return ids, y, p

def find_thresholds(y, p, grid=1001):
    # 임계값 후보
    ts = np.linspace(0, 1, grid)
    # Youden J 최적
    best_j, best_t_j = -1.0, 0.5
    # F1 최적 (positive=AD)
    best_f1, best_t_f1 = -1.0, 0.5
    for t in ts:
        pred = (p >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
        sens = tp / (tp + fn) if (tp+fn)>0 else 0.0  # Recall(AD)
        spec = tn / (tn + fp) if (tn+fp)>0 else 0.0  # Specificity(Control)
        j = sens + spec - 1
        if j > best_j:
            best_j, best_t_j = j, t
        # F1
        prec = tp / (tp + fp) if (tp+fp)>0 else 0.0
        rec  = sens
        f1 = (2*prec*rec)/(prec+rec) if (prec+rec)>0 else 0.0
        if f1 > best_f1:
            best_f1, best_t_f1 = f1, t
    return {"youdenJ": (best_t_j, best_j), "f1": (best_t_f1, best_f1)}

def evaluate_at_threshold(y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    # per-class metrics (pos=AD=1)
    prec, rec, f1, _ = precision_recall_fscore_support(y, pred, labels=[1,0], zero_division=0)
    # 지표 요약
    out = {
        "AUC": roc_auc_score(y, p),
        "AP": average_precision_score(y, p),
        "NLL": log_loss(y, np.clip(p,1e-6,1-1e-6)),
        "ACC": accuracy_score(y, pred),
        "Sensitivity(AD)": rec[0],         # AD 재현율
        "Specificity(Control)": rec[1],    # Control 재현율
        "Precision(AD)": prec[0],
        "F1(AD)": f1[0],
        "MCC": matthews_corrcoef(y, pred),
        "Threshold": t
    }
    cm_df = pd.DataFrame(
        [[tn, fp],[fn, tp]],
        index=pd.Index(["True Control(0)","True AD(1)"], name="Actual"),
        columns=pd.Index(["Pred Control(0)","Pred AD(1)"], name="Predicted")
    )
    return out, cm_df, pred

def main():
    # 1) 로드
    if os.path.exists(VAL_CSV):
        ids_v, y_v, p_v = load_probs(VAL_CSV)
        th = find_thresholds(y_v, p_v)
        t_use = th["youdenJ"][0]  # 기본: Youden J 최적 임계값
    else:
        print("※ 검증셋 파일이 없어 임계값 0.5를 사용합니다.")
        th, t_use = None, 0.5

    ids_t, y_t, p_t = load_probs(TEST_CSV)

    # 2) 테스트셋 평가
    metrics, cm_df, pred_t = evaluate_at_threshold(y_t, p_t, t_use)

    # 3) 저장
    # thresholds 기록
    with open(os.path.join(OUT_DIR, "thresholds.txt"), "w", encoding="utf-8") as f:
        if th is not None:
            f.write(f"YoudenJ_best_threshold = {th['youdenJ'][0]:.4f}, J = {th['youdenJ'][1]:.4f}\n")
            f.write(f"F1_best_threshold     = {th['f1'][0]:.4f}, F1 = {th['f1'][1]:.4f}\n")
        f.write(f"USED_threshold        = {t_use:.4f}\n")

    # 혼동행렬 표
    cm_path = os.path.join(OUT_DIR, "confusion_matrix_test.csv")
    cm_df.to_csv(cm_path)
    print("[OK] confusion matrix →", cm_path)

    # 지표 표(한 줄 요약)
    met_path = os.path.join(OUT_DIR, "metrics_test.csv")
    pd.DataFrame([metrics]).to_csv(met_path, index=False)
    print("[OK] metrics table →", met_path)

    # 샘플별 예측 표
    per_path = os.path.join(OUT_DIR, "per_sample_test.csv")
    df_per = pd.DataFrame({
        "id": ids_t,
        "label": y_t,
        "label_name": np.where(y_t==1, "AD", "Control"),
        "prob_omics": p_t,
        "pred_label": pred_t,
        "pred_name": np.where(pred_t==1, "AD", "Control")
    })
    df_per.to_csv(per_path, index=False)
    print("[OK] per-sample table →", per_path)

    # 콘솔 요약
    print("\n=== TEST summary ===")
    for k,v in metrics.items():
        print(f"{k:>20}: {v:.4f}" if isinstance(v,(int,float)) else f"{k:>20}: {v}")
    print("\nConfusion Matrix (rows=Actual, cols=Pred):")
    print(cm_df)

if __name__ == "__main__":
    main()


[OK] confusion matrix → C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\feature_build\evaluation\confusion_matrix_test.csv
[OK] metrics table → C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\feature_build\evaluation\metrics_test.csv
[OK] per-sample table → C:\Users\rhtmd\OneDrive\바탕 화면\GSE33000_results\feature_build\evaluation\per_sample_test.csv

=== TEST summary ===
                 AUC: 0.9778
                  AP: 0.9872
                 NLL: 0.1630
                 ACC: 0.9577
     Sensitivity(AD): 0.9787
Specificity(Control): 0.9167
       Precision(AD): 0.9583
              F1(AD): 0.9684
                 MCC: 0.9051
           Threshold: 0.2810

Confusion Matrix (rows=Actual, cols=Pred):
Predicted        Pred Control(0)  Pred AD(1)
Actual                                      
True Control(0)               22           2
True AD(1)                     1          46
